# オープンキャンパス管理システム — Colab + ngrok 起動ノートブック

このノートブックは Google Colab 上でアプリを起動し、**ngrok** で外部公開URLを発行します。
上から順にセルを実行してください。

> **事前準備**: ngrok は無料でも authtoken が必要です。
> https://dashboard.ngrok.com/get-started/your-authtoken で取得しておいてください（無料登録）。

## 1. 必要ライブラリのインストール

In [ ]:
!pip install flask flask-sqlalchemy pytz pyngrok -q
print("✅ ライブラリのインストール完了")

## 2. リポジトリの取得（最新コードを GitHub から取得）

初回は `git clone`、2回目以降は `git pull` で最新に更新します。

In [ ]:
import os

REPO_URL = "https://github.com/noirelumiere00/TIUkanri.git"
BRANCH = "claude/admiring-tesla-Hy3l8"
PROJECT_DIR = "/content/TIUkanri"

if not os.path.exists(PROJECT_DIR):
    !git clone -b $BRANCH $REPO_URL $PROJECT_DIR
else:
    !cd $PROJECT_DIR && git pull origin $BRANCH

os.chdir(PROJECT_DIR)
print("✅ 作業ディレクトリ:", os.getcwd())
print("ファイル一覧:", os.listdir('.'))

## 3. ngrok の authtoken を設定

実行するとトークンの入力欄が出ます（入力内容は画面に表示されません）。

In [ ]:
import getpass
from pyngrok import ngrok

token = getpass.getpass('ngrok authtoken を貼り付けて Enter: ')
ngrok.set_auth_token(token)
print("✅ authtoken を設定しました")

## 4. アプリ起動 + 公開URLの発行

公開URLが表示されたら、それをブラウザで開いてください。
**このセルは起動中ずっと実行されたままになります**（停止するには停止ボタン）。

In [ ]:
import sys
from pyngrok import ngrok

# 既存のトンネル / キャッシュをクリア（再実行時の二重起動防止）
ngrok.kill()
for m in list(sys.modules.keys()):
    if m == 'app' or m == 'models':
        del sys.modules[m]

PORT = 5000
public_url = ngrok.connect(PORT)
print("=" * 50)
print("🌐 公開URL:", public_url.public_url)
print("=" * 50)

from app import app
# Colab ではリローダーを無効にして起動する
app.run(port=PORT, use_reloader=False)

---
### 補足

- データベース `opencampus.db` はプロジェクト直下に自動生成されます。
  Colab のランタイムが切れると消えるため、永続化したい場合は Google Drive を
  マウントして `PROJECT_DIR` を Drive 内に置いてください。
- 公開URLは無料プランでは起動のたびに変わります。
- トンネルを手動で閉じるには `from pyngrok import ngrok; ngrok.kill()` を実行します。